# Play with Graph Features

This notebook demonstrates building a bipartite graph, extracting graph features, and comparing model performance.

In [ ]:
import sys
from pathlib import Path

phase_dir = Path().cwd().parent
sys.path.insert(0, str(phase_dir / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from phase01_lightgbm_baseline.data_loader import DataLoader
from phase01_lightgbm_baseline.lightgbm_baseline import LightGBMWrapper
from phase01_lightgbm_baseline.graph_features import GraphFeatureExtractor
from phase01_lightgbm_baseline.comparison import compute_comparison_report

## Load Data

In [ ]:
loader = DataLoader()
train_df, test_df = loader.load_data(sample_limit=500)

print(f"Train samples: {len(train_df)}")
print(f"Test samples: {len(test_df)}")

## Build Graph

In [ ]:
extractor = GraphFeatureExtractor(user_col="card1", merchant_col="ProductCD")
G = extractor.build_bipartite_graph(train_df)

print(f"Graph statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Users: {sum(1 for n in G.nodes() if G.nodes[n].get('node_type') == 'user')}")
print(f"  Merchants: {sum(1 for n in G.nodes() if G.nodes[n].get('node_type') == 'merchant')}")

## Extract Graph Features

In [ ]:
train_graph, test_graph = extractor.append_graph_features(train_df.copy(), test_df.copy())

graph_feature_cols = [
    "user_degree", "user_clustering", "user_degree_centrality", "user_community",
    "user_rwse_length", "user_rwse_unique", "user_rwse_degree",
    "merchant_degree", "merchant_clustering", "merchant_degree_centrality",
    "merchant_rwse_length", "merchant_rwse_unique", "merchant_rwse_degree",
]

print(f"Graph features added: {len(graph_feature_cols)}")
print(f"Train shape: {train_graph.shape}")
print(f"Test shape: {test_graph.shape}")

## Train Baseline Model

In [ ]:
baseline_features = ["TransactionAmt", "time_since_creation", "n_prior_transactions"]

X_train_base = train_df[baseline_features].values
y_train = train_df["isFraud"].values
X_test_base = test_df[baseline_features].values
y_test = test_df["isFraud"].values

baseline_model = LightGBMWrapper(random_state=42)
baseline_metrics = baseline_model.full_pipeline(X_train_base, y_train, X_test_base, y_train, X_test_base, y_test, k=10)

print("Baseline metrics:")
for metric, value in baseline_metrics.items():
    print(f"  {metric}: {value:.4f}")

## Train Graph-Aware Model

In [ ]:
all_features = baseline_features + graph_feature_cols

X_train_graph = train_graph[all_features].values
X_test_graph = test_graph[all_features].values

graph_model = LightGBMWrapper(random_state=42)
graph_metrics = graph_model.full_pipeline(X_train_graph, y_train, X_test_graph, y_train, X_test_graph, y_test, k=10)

print("Graph-aware metrics:")
for metric, value in graph_metrics.items():
    print(f"  {metric}: {value:.4f}")

## Compare Models

In [ ]:
deltas = compute_comparison_report(baseline_metrics, graph_metrics)

print("Delta metrics (graph - baseline):")
for metric, value in deltas.items():
    print(f"  {metric}: {value:+.4f}")

## Visualize Comparison

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_to_plot = ["pr_auc", "roc_auc", "brier_score"]
baseline_vals = [baseline_metrics[m] for m in metrics_to_plot]
graph_vals = [graph_metrics[m] for m in metrics_to_plot]

x = np.arange(len(metrics_to_plot))
width = 0.35

axes[0].bar(x - width/2, baseline_vals, width, label='Baseline', alpha=0.7)
axes[0].bar(x + width/2, graph_vals, width, label='Graph', alpha=0.7)
axes[0].set_ylabel('Metric Value')
axes[0].set_title('Model Comparison')
axes[0].set_xticks(x)
axes[0].set_xticklabels(metrics_to_plot)
axes[0].legend()

axes[1].bar(range(len(graph_feature_cols)), 
            [train_graph[col].mean() for col in graph_feature_cols],
            edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Graph Feature')
axes[1].set_ylabel('Mean Value')
axes[1].set_title('Graph Feature Statistics')
axes[1].set_xticks(range(len(graph_feature_cols)))
axes[1].set_xticklabels([c.replace('user_', '').replace('merchant_', '') for c in graph_feature_cols], rotation=45, ha='right')

plt.tight_layout()
plt.show()

## Summary

We built a bipartite graph, extracted graph features, and compared baseline vs graph-aware models. The delta metrics show whether graph features improve performance.